# Modul 2: Uji Stasioneritas (ADF/KPSS), Diferensiasi, & Correlogram
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 2 (Minggu 2):** Menguji stasioneritas kovarians menggunakan uji ADF dan KPSS, menerapkan diferensiasi, serta menginterpretasikan plot ACF/PACF.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings('ignore')
plt.rcParams['axes.grid'] = True

## 1. Pembangkitan Sinyal Daya Aktif Transmisi (Non-Stasioner)

In [ ]:
rng = np.random.default_rng(2026)
n = 300
t = np.arange(n)
# Random walk + drift + komponen AR(1)
eps = rng.normal(0, 3.0, n)
power = 150 + np.cumsum(0.25 + eps)
s = pd.Series(power, index=pd.date_range('2026-01-01', periods=n, freq='h'), name='active_power_mw')

s.plot(figsize=(12, 4), color='navy', title='Profil Daya Aktif Saluran Transmisi (Data Mentah)')
plt.ylabel('Daya Aktif (MW)')
plt.show()

## 2. Fungsi Uji Gabungan ADF + KPSS (4 Kuadran Stasioneritas)

In [ ]:
def check_stationarity_full(series, label="Sinyal"):
    clean = series.dropna()
    adf_stat, adf_p, *_ = adfuller(clean, autolag='AIC')
    kpss_stat, kpss_p, *_ = kpss(clean, regression='c', nlags='auto')
    print(f"=== Uji Stasioneritas: {label} ===")
    print(f"  ADF  Stat = {adf_stat:8.4f} | p-value = {adf_p:.4e} (H0: Non-Stasioner)")
    print(f"  KPSS Stat = {kpss_stat:8.4f} | p-value = {kpss_p:.4f}     (H0: Stasioner)")
    if adf_p < 0.05 and kpss_p > 0.05:
        print("  -> Status: STASIONER MURNI (Kuadran I)")
    elif adf_p >= 0.05 and kpss_p <= 0.05:
        print("  -> Status: NON-STASIONER / DSP (Wajib Diferensiasi)")
    elif adf_p >= 0.05 and kpss_p > 0.05:
        print("  -> Status: TREND-STATIONARY / TSP (Butuh Detrending)")
    else:
        print("  -> Status: PERINGATAN (Indikasi Structural Break)")

check_stationarity_full(s, "Data Mentah (d=0)")
s_diff1 = s.diff().dropna()
check_stationarity_full(s_diff1, "Setelah Diferensiasi Ordo-1 (d=1)")

## 3. Visualisasi Correlogram (ACF & PACF)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
plot_acf(s, lags=30, ax=axes[0, 0], title='ACF Data Mentah (Meluruh Lambat)')
plot_pacf(s, lags=30, ax=axes[0, 1], title='PACF Data Mentah')
plot_acf(s_diff1, lags=30, ax=axes[1, 0], title='ACF Setelah Diferensiasi d=1')
plot_pacf(s_diff1, lags=30, ax=axes[1, 1], title='PACF Setelah Diferensiasi d=1')
plt.tight_layout()
plt.show()

## 4. Tugas & Pertanyaan Analisis (C3–C6)
1. **[C3]** Lakukan diferensiasi ordo-2 (`s.diff().diff()`) dan bandingkan variansinya dengan `s_diff1`. Apakah terjadi *over-differencing*?
2. **[C4]** Jelaskan perbedaan visual plot ACF sebelum dan sesudah diferensiasi!
3. **[C6]** Tulis fungsi `auto_stationary(series, max_d=2)` yang secara otomatis menaikkan ordo $d$ sampai lolos uji ADF ($p < 0.05$) dan KPSS ($p > 0.05$).